## **PPG_Signal_Measurement_Implementation_Unit(II)**

> In this experiment, we will learn how to practically use derivative on signal

> Using it with the time difference of two point detected using ECG peak detection to get Pulse Arrival Time (PAT)!

> Finally, use least square method in statistic to estimate your systolic blood pressure!

# 0. First import some formulas and functions we will be using!

In [ ]:
#@title
from google.colab import files
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.signal import find_peaks
import math
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
def find_peak_weird_detection(data, height, distance):
  peak_list = find_peaks(data, height=height, distance=distance)[0]
  if(peak_list.size != 0):
    print('Found a total of', peak_list.size, 'peak')
    peak_peak_list = (np.diff(peak_list))
    pp_mean = np.mean(peak_peak_list)
    counter = 0
    counter2 = 0
    counter3 = 0

    for i in peak_peak_list:
      if(i > pp_mean+100):
        counter2 = counter2 + 1

    print('Found',counter2,'[weird peak]')

    weird_peak_list_start = np.zeros(counter2,int)
    weird_peak_list_end = np.zeros(counter2,int)
    weird_pp_list = np.zeros(counter2,int)


    for i in peak_peak_list:
      if(i > pp_mean+100):
        weird_peak_list_start[counter3] = peak_list[counter]
        weird_peak_list_end[counter3] = peak_list[counter+1]
        weird_pp_list[counter3] = i
        counter3 += 1
      counter = counter + 1

    pp_mean_fixed_1 = (pp_mean*(peak_peak_list.size))
    pp_mean_fixed_2 = pp_mean_fixed_1 - np.sum(weird_pp_list)
    pp_mean_fixed = int(pp_mean_fixed_2/(peak_peak_list.size - weird_pp_list.size))

  else:
    print('Unable to find any peak, please readjust the height and distance to find the peak')

  return peak_list, peak_peak_list, weird_peak_list_start, weird_peak_list_end, pp_mean_fixed

def cut_first_data(data):
  data = data[1:]
  return data

# 1. Enter the sample ECG signal and sample PPG signal into the codebase!

You can select both file at once

In [ ]:
uploaded = files.upload()

for fn in uploaded.keys():
  print('You have uploaded','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2. Look at what the ECG signal and PPG signal looks like!

In [ ]:
#Let's read the uploaded file!
df = pd.read_csv('Sample_ECG_Signal.txt') #Remember to change to your own file name!
a1 = np.array(df)
a2 = a1[0 : , 0]

#Let's read the uploaded file!
df2 = pd.read_csv('Sample_PPG_Signal.txt') #Remember to change to your own file name!
b1 = np.array(df2)
b2 = b1[0 : , 0]

In [ ]:
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=a2,
    name='Original Plot'
))
fig.update_layout(
    title="Synchronously measured ECG Signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your ECG signal!
fig.show()

In [ ]:
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b2,
    name='Original Plot'
))
fig.update_layout(
    title="Synchronously measured PPG Signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your PPG signal!
fig.show()

# 3. Using peak detection method let's first get the peaks of the ECG signal!

> If after execution you see some weird peak, try adjusting the height and distance!

> Observe the ECG signal above. height adjustment can reference average peak height on the y-axis.

> distance adjustment can reference the the approximate distance between the peak on the x-axis.

In [ ]:
height = 63
distance = 600
(peak_list_a2, pp_list_a2, weird_p_start_a2, weird_p_end_a2, pp_mean_a2) = find_peak_weird_detection(a2, height, distance)

In [ ]:
#Test the detected waveform again!
length = 40000
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=a2[0:length],
    mode='lines',
    name='ECG Signal'
))

counter = 0
for i in peak_list_a2:
  counter += 1
  if(i >= length):
    break

#Plot the detected peaks on the figure as well!
fig.add_trace(go.Scatter(
    x=peak_list_a2[0:counter],
    y=[a2[j] for j in peak_list_a2],
    mode='markers',
    marker=dict(
        size=8,
        color='red',
        symbol='cross'
    ),
    name='Detected Peak Position'
))
fig.update_layout(
    title="Synchronously measured ECG Signal and Peak Detection result",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your result!
fig.show()

# 4. Use So and Chan algorithm to calculate PPG signal's slope on each point. Then use peak detection to find the peaks of these slope!

> Slope's main concept is to understand the rate of change. The common two-point method (the point minus the previous point).

> So and Chan algorithm strengthen this concept. Delayed two-point method, which consider two point both in front and behind as well!

> The formula is Slope(n) = -2*X(n-2) - 1*X(n-1) + 1*X(n+1) + 2*X(n+2)

In [ ]:
counter = 2 #We don't have data for X(n-2) if we start at the beginning. Therefore we need to start from the third point

b3 = np.zeros(np.size(b2)-2) #Creating an empty space to put the slopes of PPG signal
while(counter < np.size(b2)-2):
  b3[counter] = (-2)*b2[counter-2] + (-1)*b2[counter-1] + 1*b2[counter+1] + 2*b2[counter+2]
  counter += 1

NameError: name 'np' is not defined

In [ ]:
#Test the detected waveform again!
length2 = 40000
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b3[0:length],
    mode='lines',
    name='Original Plot'
))
fig.update_layout(
    title="PPG slope calculated using So and chan",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your result!
fig.show()

> If after execution you see some weird peak, try adjusting the height and distance!

> Observe the PPG signal above. height adjustment can reference average peak height on the y-axis.

> distance adjustment can reference the the approximate distance between the peak on the x-axis.

In [ ]:
height2 = 4
distance2 = 600
(peak_list_b3, pp_list_b3, weird_p_start_b3, weird_p_end_b3, pp_mean_b3) = find_peak_weird_detection(b3, height2, distance2)

In [ ]:
#Test the detected waveform again!
length2 = 40000
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=b3[0:length2],
    mode='lines',
    name='PPG Signal\'s So and chan Slope'
))

counter2 = 0
for i in peak_list_b3:
  counter2 += 1
  if(i >= length2):
    break

#Plot the detected peaks on the figure as well!
fig.add_trace(go.Scatter(
    x=peak_list_b3[0:counter2],
    y=[b3[j] for j in peak_list_b3],
    mode='markers',
    marker=dict(
        size=8,
        color='red',
        symbol='cross'
    ),
    name='Detected Peak Position'
))

fig.update_layout(
    title="Synchronously measured PPG Signal slope and Peak Detection result",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your result!
fig.show()

# 5. Confirm ECG and PPG peak data

> Why do we need to confirm the peak data? It isn't because we are worried that there are bugs in the code but because we are worried that the data is incomplete!

> Think of the instant you hit [record], the ECG peak has just passed by, but the PPG slope peak just arrived! Or that instant is when you hit [stop recording].

> In these two circumstances it will cause the data to have one extra point or one less point, or unable to sync up the points. While we could use more complex algorithm to solve this, for us beginner let's just start with observing it manually!

In [ ]:
#Manually observe ECG signal and PPG signal's slope's detailed information

print('ECG signal\'s peak position is at',peak_list_a2)
print('PPG signal\'s slope\'s peak position is at',peak_list_b3)
print('ECG signal\'s peak amount is',np.size(peak_list_a2))
print('PPG signal\'s slope\'s peak amount is',np.size(peak_list_b3))

# Caution!!!!!!! Do not execute the following code code unless absolutely necessary!!!!!

The following code will remove the first data point of the PPG signal

In [ ]:
#Caution!!!!!!! Do not execute the following code code unless absolutely necessary!!!!!
#Only need to run this code if the first PPG data is bigger than the first ECG data
#Observe that for the sample signal the first PPG signal is 245, the first ECG signal is 480
#If we conclude that the data is valid then we do not need to execute this code
peak_list_b3 = cut_first_data(peak_list_b3) #Delete the first data point of the PPG signal

In [ ]:
#Check the ECG signal and PPG signal's slope's detailed information again

print('ECG signal\'s peak position is at',peak_list_a2)
print('PPG signal\'s slope\'s peak position is at',peak_list_b3)
print('ECG signal\'s peak amount is',np.size(peak_list_a2))
print('PPG signal\'s slope\'s peak amount is',np.size(peak_list_b3))

#6. Formal analysis of Pulse Arrival Time (PAT) data

In [ ]:
#First set Pulse Arrival Time's value to prevent two data's value being different
#Because we are subtracting one data set from the other, if one set has more data, take the lesser one

PAT_list_number = 0
if(np.size(peak_list_b3) >= np.size(peak_list_a2)):
  PAT_list_number = np.size(peak_list_a2)
else:
  PAT_list_number = np.size(peak_list_b3)

print('Set Pulse Arrival Time\'s value as', PAT_list_number)

In [ ]:
PAT_list = np.zeros(PAT_list_number) #Create space for PAT data according to the value that was set above
counter = 0
while counter < PAT_list_number:
    print('Subtracting',counter+1,'\'s peak from one another gives a PAT of',peak_list_b3[counter] - peak_list_a2[counter],'ms')
    PAT_list[counter] = peak_list_b3[counter] - peak_list_a2[counter]
    counter = counter + 1

PAT_mean = np.mean(PAT_list)
print("\n") #Set new line
print("The average of all PATdata is", round(PAT_mean,2),'ms')

# 6. According to Pulse Arrival Time's formula simulated beforehand to estimate the blood pressure

> Simulating formula isn't difficult. All you need is to pair a blood pressure monitor when recording PAT and do regression calculation with both data

> Because you probably don't have blood pressure monitor beside you, therefore we will use this sample data as an example

> Blood pressure estimation linear equation: BP = A * (d^2 / PAT^2) + B

In [ ]:
#Below is ten sample data points measured with blood pressure monitor beforehand. When measuring blood pressure at 110mm/Hg, PAT is 241.38 ms, and so on
#If you want to create your own model, can first measure blood pressure and immediately measure ECG/PPG right after
#After getting several set of data through repeated measurement, use the code in the previous section to calculate the PAT of different blood pressure, then change the data here to the corresponding data

BP = np.array([110, 127, 115, 120, 131, 125, 117, 121, 120, 126])

PAT = np.array([241.38, 228.85, 233.43, 223.22, 214.45, 225.2, 218.68, 219.19, 255, 236.93]) #單位：毫秒

In [ ]:
#Create the variable need for the linear function BP = A * (d^2 / PAT^2) + B

secPAT = PAT*0.001                 #Set the PAT unit to "second"
length = 1.75                    #Unit: meter you can set it to your own height here!
d2PAT2 = pow(length*0.6 ,2) / pow(secPAT, 2) #Multiply height to arm length ratio of 0.6, and calculate with PAT

print(d2PAT2)

In [ ]:
#Before doing regression calculation, first use the figure to get a feeling of the data's correlation!

def create_screen():
  # create a figure and axes
  fig = plt.figure(figsize=(12,5))
  ax1 = plt.subplot(1,2,1)

  # set up the subplots as needed
  ax1.set_xlim(( 0, 40))
  ax1.set_ylim((80, 200))
  ax1.set_xlabel('d^2 / PAT^2')
  ax1.set_ylabel('Blood Pressure')

  txt_title = ax1.set_title('')

  BP_plot, = ax1.plot(d2PAT2, BP,   'o', color = 'blue')

  ax1.legend(['BP_plot']);
  return fig,ax1,txt_title,BP_plot
create_screen()

>From the figure above we can see that blood pressure and the data are positively correlated

In [ ]:
#Do regression calculation with BP and(d^2 / PAT^2)!
lm = LinearRegression()
lm.fit(np.reshape(d2PAT2, (len(d2PAT2), 1)), np.reshape(BP, (len(BP), 1)))

# Print out coefficient
print(lm.coef_)

# Print out intercept
print(lm.intercept_)


print('The linear function of the regression calculation is BP=',lm.coef_[0,0],'* d2PAT2 +',lm.intercept_[0] )

In [ ]:
#Then we just need to substitute the average PAT value of this measurement into the equation to derive your estimated blood pressure!

print('For this measurement the average PAT is',round(PAT_mean,2) ,'ms')
secPAT_mean = PAT_mean * 0.001 #Change unit to 'second'
d2PAT2_this_time = pow(length*0.6 ,2) / pow(secPAT_mean, 2)

BP_estimate = (lm.coef_[0,0] * d2PAT2_this_time) + lm.intercept_[0]

print('The estimated blood pressure is',round(BP_estimate,2) ,'mm/Hg')

此次量測算出的平均PAT為 221.18 毫秒
此次估算的血壓為 122.9 mm/Hg


# Chapter Summary
---
1. The instant blood arrive at the finger we first need to derive the PPG signal then get the peak position.
2. Blood pressure and PAT have a certain amount of correlation.
3. To get the most accurate blood pressure equation, each person need to first use blood pressure monitor's data to fine-tune the equation's coefficient and intercept.


# Congratulation on completing this chapter's classes



> For student that needs the class certificate you can use the password on the bookstore website to obtain it

> Certificate password: [yutechEOGPPG3]!